## Interrogatorio a los Datos

El Reto y Contexto: Siguen trabajando para la aseguradora marítima. Ya saben cómo obtener la radiografía general (.describe()), pero el director de riesgos necesita respuestas segmentadas y sospecha que hubo fraudes en las tarifas cobradas. Deben usar técnicas avanzadas de sumarización en memoria para confirmarlo.


Instrucciones Técnicas: Abran un Jupyter Notebook nuevo carguen el dataset crudo directamente desde el repositorio:

In [3]:
import pandas as pd
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

# Pregunta A Cuantos sobrevivieron y cuantos murieron en el naufragio del Titanic?
df['Survived'].value_counts(normalize=True)

# Pregunta B Cual fue la tasa de supervivencia por sexo?
df.groupby('Sex')['Survived'].value_counts(normalize=True).unstack()

# Pregunta C Cual fue la cantidad de outliers por clase?
fare_IQR = df["Fare"].quantile(0.75) - df["Fare"].quantile(0.25)

outlier_count = 0
class_outlier = {
    1: 0,
    2: 0,
    3: 0
}

for row in df.iterrows():
    if row[1]['Fare'] < (df["Fare"].quantile(0.25) - 1.5 * fare_IQR) or row[1]['Fare'] > (df["Fare"].quantile(0.75) + 1.5 * fare_IQR):
        outlier_count += 1
        row_class = row[1]['Pclass']
        class_outlier[row_class] += 1

print("Number of outliers in Fare:", outlier_count)
print("Outliers by class:", class_outlier)

# Pregunta D media y medianana de fare
print("Media de Fare:", df["Fare"].mean())
print("Mediana de Fare:", df["Fare"].median())

# Pregunta E Muestra de 150 registros con proporcion identica de sobrevivientes y no sobrevivientes de la db
balanced_sample = df.groupby('Survived', group_keys=False).apply(lambda x: x.sample(min(len(x), 75)))

# Pregunta F sesgo 
df.groupby('Survived')['Age'].mean()

# Pregunta H Titulos con expreciones regulares
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
display(df.groupby('Title')['Age'].median())

# Pregunta I Varianza Survived
varianza_survived = df['Survived'].var()
print(f"Varianza de Survived: {varianza_survived}")

# Pregunta J Agrupacion por niveles
display(df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count())


<>:43: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:43: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/4f/vhqpbq4d7nx_9k5xcy2j8jgm0000gn/T/ipykernel_4320/1436010332.py:43: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


Number of outliers in Fare: 116
Outliers by class: {1: 104, 2: 5, 3: 7}
Media de Fare: 32.204207968574636
Mediana de Fare: 14.4542


Title
Capt        70.0
Col         58.0
Countess    33.0
Don         40.0
Dr          46.5
Jonkheer    38.0
Lady        48.0
Major       48.5
Master       3.5
Miss        21.0
Mlle        24.0
Mme         24.0
Mr          30.0
Mrs         35.0
Ms          28.0
Rev         46.5
Sir         49.0
Name: Age, dtype: float64

Varianza de Survived: 0.23677221654749742


Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

**Pregunta A (Sumarización Categórica): Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?**
61.61% no sobrevivieron
38.38% Si sobrevivio

**Pregunta B (Agrupación y Agregación): El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?**
74.20% de las mujeres sobrevivieron y solo 18.89% de los hombres sobrevivieron


**Pregunta C: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?**
Outliers by class: {1: 104, 2: 5, 3: 7}


**Pregunta D: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?**
La media es mas grande por la cantidad de outliers tan grandes al usar knn podriamos enfrentar que los outliers afecten los resultados por diferencias grandes entre vecinos


**Pregunta E: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?**
Evitamos el no reflejar la proporcion del data set real


**Pregunta F: Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?**
Estamos haciendo que nuestro nueva muestra no refleje cosas de la proporcion del data set entero

**Pregunta G: Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.**
No deberiamos eliminarlos por que a pesar de ser outliers pueden justificados por ser casos especificos tipo los de primera clase tenian un barco de rescate en su cabina y eliminarlos pueden perder datos valiosas

**Pregunta H: Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?**
Usar la mediana es mejor por que agrupomaos los individuos por caracteristicas que los relacionan con la edad, un MR va a rondar los 40 años si usamos la mediana flobal para los titulos el algoritmo clasificaria niños como adultos pero al usar subgrupos basados en los titulos reducimos el sesgo


**Pregunta I: Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?**  Si la varianza fuera 0 significa que no hay dispercion en los datos, todos los pasajeros o murieron o sobrevivieron si entrenamos un algoritmo asi el modelo falla o prediciria siempre el mismo resultado sin importar los datos


**Pregunta J: Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?**  Con estos grupos va a haber overfitting el algoritmo aprende de un grupo con 1 o 2 pasajeros y no hay una señal fuerte que este grupo realmente muestre un patron importante y memorizara datos que no son mas que ruido, el modelo tiene que aprender a generalizar para grupos pequeños como esos